# Dataset Preparation

**Run this notebook once on Colab** to generate the Q-factor CSV and save it to Google Drive.
After it finishes, copy the file ID from the Drive link and set `GDRIVE_FILE_ID` in your analysis notebooks.

Steps:
1. Mount Google Drive
2. Download & extract raw z-map data
3. Compute Q-factors (GPU-accelerated if available)
4. Save CSV to Drive

In [ ]:
import os
import sys
import zipfile
import shutil
import subprocess
from pathlib import Path

import gdown

REPO_ROOT = Path('/content/ml-parameter-optimization')

if not REPO_ROOT.exists():
    print('Cloning repository...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/arayit/ml-parameter-optimization.git', str(REPO_ROOT)],
        check=True,
    )

DRIVE_URLS = {
    '06082025': 'https://drive.google.com/file/d/1BN4DzT8Pl-FMxYl0oJ0sPYHPEQ4pjf7X/view?usp=sharing',
    '26082025': 'https://drive.google.com/file/d/1kh9HPNOR-mPc1F9yKjheTS2jZrak4Vek/view?usp=sharing',
}
DATA_DIR  = REPO_ROOT / 'data' / 'dataset_combined'
CSV_PATH  = REPO_ROOT / 'qfactors_1.0um.csv'
DRIVE_CSV = '/content/drive/MyDrive/AI Material Processing Data/qfactors_1.0um.csv'

print(f'Repo root: {REPO_ROOT}')

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Download & Extract Raw Data

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

for dataset, url in DRIVE_URLS.items():
    zmap_dir = DATA_DIR / dataset / 'zmap'
    if zmap_dir.exists() and any(zmap_dir.iterdir()):
        print(f'  {dataset}: already extracted, skipping.')
        continue
    zip_path = REPO_ROOT / f'{dataset}.zip'
    if not zip_path.exists():
        print(f'  Downloading {dataset}.zip ...')
        gdown.download(url=url, output=str(zip_path), fuzzy=True, quiet=True)
    print(f'  Extracting {dataset}.zip ...')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

print('Done.')

## Step 3 — Compute Q-Factors

In [ ]:
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True)
    use_gpu = result.returncode == 0
except FileNotFoundError:
    use_gpu = False

print(f'GPU {"detected" if use_gpu else "not available"} — running on {"GPU" if use_gpu else "CPU"}.')

if use_gpu:
    print('Installing CuPy for GPU acceleration...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], check=True)

env = os.environ.copy()
env['TQDM_DISABLE'] = '1'
if use_gpu:
    env['USE_GPU'] = '1'

result = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'src' / 'extract_qfactors.py')],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'Extraction failed (exit {result.returncode})')

print('Q-factors computed.')

## Step 4 — Save CSV to Google Drive

In [ ]:
os.makedirs(os.path.dirname(DRIVE_CSV), exist_ok=True)
shutil.copy(CSV_PATH, DRIVE_CSV)
print(f'Saved to: {DRIVE_CSV}')
print()
print('Next steps:')
print('  1. Open Google Drive → AI Material Processing Data')
print('  2. Right-click qfactors_1.0um.csv → Share → Copy link')
print('  3. Extract the file ID from the link')
print('  4. Paste it into GDRIVE_FILE_ID in your analysis notebooks')